In [1]:
!pip install plotly numpy pandas scikit-learn joblib

In [2]:
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

import os
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

from hearing_box import (
    Box,
    compute_A2_hat,
)

PI = math.pi

In [3]:
# geometry
A = 1.0

# spectral truncation
N_EIGS = 2000

# f scan
F_RANGE = range(1, 201)

# heat trace fit
TMAX_FACTOR = 500.0
M_GRID = 251

# grid resolution
N_SAMPLES = 200

# parallelization
N_JOBS = 12

# domain of aspect ratios
R_MIN, R_MAX = 0.3, 3.0
S_MIN, S_MAX = 0.3, 3.0

## Grid Erzeugen

In [4]:
r_vals = np.geomspace(R_MIN, R_MAX, N_SAMPLES)
s_vals = np.geomspace(S_MIN, S_MAX, N_SAMPLES)

## Funktionen

In [5]:
def box_eigenvalues(a: float, b: float, c: float, N: int, oversample: float = 1.25, max_iters: int = 8):
    V = a * b * c
    lam_cut = (6 * PI**2 * N / V) ** (2.0 / 3.0) * oversample

    a2, b2, c2 = a * a, b * b, c * c
    vals = None

    for _ in range(max_iters):
        n_max = math.ceil(a * math.sqrt(lam_cut) / PI)
        m_max = math.ceil(b * math.sqrt(lam_cut) / PI)
        k_max = math.ceil(c * math.sqrt(lam_cut) / PI)

        n = np.arange(1, n_max + 1, dtype=np.float64)
        m = np.arange(1, m_max + 1, dtype=np.float64)
        k = np.arange(1, k_max + 1, dtype=np.float64)

        vals = PI**2 * (
            (n[:, None, None] ** 2) / a2
            + (m[None, :, None] ** 2) / b2
            + (k[None, None, :] ** 2) / c2
        )

        vals = np.sort(vals.ravel())

        if vals.size >= N:
            return vals[:N]

        lam_cut *= 1.35

    return vals

In [6]:
def stability_minimum_from_eigs(a: float, b: float, c: float, all_eigs, N: int = 2000, f_range=range(10, 121), tmax_factor: float = 500.0, m_grid: int = 251):
    eigs_sub = np.asarray(all_eigs[:N], dtype=float)
    A2_true = (a + b + c) / (4 * PI**2)

    scan_A2 = np.array([
        compute_A2_hat(
            eigs_sub,
            f,
            tmax_factor=tmax_factor,
            m_grid=m_grid,
            diagnostics=False,
        )
        for f in f_range
    ])

    errors = np.abs(scan_A2 / A2_true - 1.0)
    best_idx = np.argmin(errors)
    return list(f_range)[best_idx]

In [7]:
def evaluate_grid_point(r, s):
    
    b = float(r)
    c = float(s)

    eigs = box_eigenvalues(A, b, c, N_EIGS)

    f_star = stability_minimum_from_eigs(
        A, b, c,
        eigs,
        N=N_EIGS,
        f_range=F_RANGE,
        tmax_factor=TMAX_FACTOR,
        m_grid=M_GRID,
    )

    return (r, s, f_star)

In [8]:
eigs = box_eigenvalues(1.0, 1.5, 2.3, 2000)
f_best = stability_minimum_from_eigs(1.0, 1.5, 2.3, eigs, N=2000, f_range=range(10, 121))
print("fBest =", f_best)

fBest = 69


## Start der Berechnung

In [9]:
grid_points = [(r, s) for r in r_vals for s in s_vals]

results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(evaluate_grid_point)(r, s)
    for r, s in grid_points
)

data = np.array(results, dtype=float)

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   1 tasks      | elapsed:    1.9s
[Parallel(n_jobs=12)]: Done   8 tasks      | elapsed:    3.3s
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    3.4s
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    5.3s
[Parallel(n_jobs=12)]: Done  37 tasks      | elapsed:    6.6s
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:    8.5s
[Parallel(n_jobs=12)]: Done  61 tasks      | elapsed:   10.0s
[Parallel(n_jobs=12)]: Done  74 tasks      | elapsed:   12.1s
[Parallel(n_jobs=12)]: Done  89 tasks      | elapsed:   14.1s
[Parallel(n_jobs=12)]: Done 104 tasks      | elapsed:   16.3s
[Parallel(n_jobs=12)]: Done 121 tasks      | elapsed:   18.6s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:   21.0s
[Parallel(n_jobs=12)]: Done 157 tasks      | elapsed:   23.9s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:   26.7s
[Parallel(n_jobs=12)]: Done 197 tasks      | elapsed:  

In [13]:
z_mat = data[:,2].reshape(len(r_vals), len(s_vals))

In [14]:
R, S = np.meshgrid(r_vals, s_vals, indexing="ij")

fig = go.Figure(
    data=[go.Surface(x=R, y=S, z=z_mat, colorscale="Viridis")]
)

fig.update_layout(
    scene=dict(
        xaxis=dict(title="b/a", type="log"),
        yaxis=dict(title="c/a", type="log"),
        zaxis=dict(title="f*"),
        aspectratio=dict(x=1,y=1,z=0.5)
    ),
    width=900,
    height=700
)

fig.show()

## Export to CSV

In [15]:
df = pd.DataFrame({
    "a": np.full_like(data[:,0], A),
    "b": data[:,0],
    "c": data[:,1],
    "x": data[:,0],
    "y": data[:,1],
    "z": data[:,2]
})
df = df.sort_values(["b","c"])
csv_file = f"stability_surface_box_N{N_EIGS}_grid{N_SAMPLES}.csv"

df.to_csv(csv_file, index=False)
print("CSV exported:", csv_file)

CSV exported: stability_surface_box_N2000_grid200.csv
